# 03 · 回测研究

**目的**:用现有回测框架(embargo walk-forward → 信号帧 → 九组策略对比)看最终效果。

流程:`models.walk_forward`(每折:训练窗内特征选择 + 5 模型集成)→
`perp_backtest.build_signal_frame` → `run_all`(默认策略/日频/多空/闸门/基准)。

**运行时间**:`WF_STEP=40` 快速模式约 5-6 分钟;出正式结论请改成 `10`(约 15-20 分钟)。

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent                      # launched from research/
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import binance_data as bd
import factors as F
import models as M

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (13, 5)
print(f"project root: {ROOT}")

In [ ]:
import perp_backtest as pb

WF_STEP = 40          # 快速探索;正式结论用 10
OUT = ROOT / 'research' / 'output'
OUT.mkdir(parents=True, exist_ok=True)

## 1. 面板 + walk-forward(耗时步骤)

In [ ]:
frames = bd.load_universe()
panel = F.build_panel(frames, verbose=True)

results = {}
for name in ['selection', 'timing']:
    preds, m = M.walk_forward(panel, name, wf_step=WF_STEP, verbose=True)
    results[name] = {'wf_predictions': preds, 'wf_metrics': m}

### 三种排名分数的 RankIC(集成均值 / t-stat 置信度 / LCB)

已知结论:t-stat 平均 IC 更高但排序不粘、不适配迟滞规则,策略默认用均值。

In [ ]:
sel = results['selection']['wf_predictions']
for col, label in [('prediction', '集成均值'), ('prediction_tstat', 't-stat 置信度'),
                   ('prediction_lcb', 'LCB')]:
    if col in sel.columns:
        r, t = M._cross_sectional_ic(sel, col, 'target_ret_7d')
        print(f"  {label:<12} RankIC {r:+.4f} (t={t:+.2f})")

## 2. 策略回测(九组对比)

In [ ]:
signals = pb.build_signal_frame(results)
signals.to_parquet(OUT / f'signals_wf{WF_STEP}.parquet')

summary, curves = pb.run_all(signals)
pb.format_summary(summary)

In [ ]:
fig = pb.plot_curves(curves, show=True)

## 3. 默认策略的细看:回撤、滚动 Sharpe、换手

In [ ]:
d = curves['Selection + Hysteresis']
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(d['Date'], (d['equity'] - 1) * 100, color='darkgreen', lw=1.5)
axes[0].set_ylabel('累计净收益 %'); axes[0].grid(ls=':', alpha=0.5)
dd = d['equity'] / d['equity'].cummax() - 1
axes[1].fill_between(d['Date'], dd * 100, 0, color='crimson', alpha=0.5)
axes[1].set_ylabel('回撤 %'); axes[1].grid(ls=':', alpha=0.5)
rs = d['net_return'].rolling(90).mean() / d['net_return'].rolling(90).std() * np.sqrt(365)
axes[2].plot(d['Date'], rs, lw=1)
axes[2].axhline(0, color='k', lw=0.6)
axes[2].set_ylabel('滚动 90d Sharpe'); axes[2].grid(ls=':', alpha=0.5)
plt.tight_layout(); plt.show()
print(f"日均换手 {d['turnover'].mean():.3f} | 最深回撤 {dd.min():.1%} | "
      f"资金费合计 {d['funding_cost'].sum():+.2%}(负=收入)")

## 4. 参数敏感度示例:top_n(重放,不重训,秒级)

In [ ]:
rows = []
for n in [3, 5, 8, 10, 15]:
    daily = pb.simulate(signals, pb.rule_selection_hysteresis,
                        top_n=n, exit_rank_mult=config.EXIT_RANK_MULT)
    m = pb.metrics(daily)
    rows.append({'top_n': n, 'Total': f"{m['Total Return']:+.1%}",
                 'Sharpe': f"{m['Sharpe']:+.2f}", 'MaxDD': f"{m['Max Drawdown']:.1%}",
                 'Turnover': f"{m['Avg Turnover']:.2f}"})
pd.DataFrame(rows)

## 结论区(手写)

- 本次数据窗口下的模型层指标 vs 历史记录(README):
- 默认策略与各对照的相对结论是否维持:
- 想推进的改动(记得先过 wf_step=10 再下结论;策略层总收益噪声极大,看相对不看绝对):